# Technology Control Tower - Interactive Notebook

This notebook allows you to run the Technology Control Tower interactively.

## Setup
Make sure you have:
1. Installed dependencies: `pip install -r requirements.txt`
2. Created `.env` and `config/config.yaml` files

Run each cell by pressing **Shift+Enter**

## 1. Import Required Libraries

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
from datetime import datetime

# Add src directory to path
sys.path.append('./src')

# Import Control Tower modules
from master_register import MasterRegister
from action_tracker import ActionTracker
from tpr_reporting import TPRReporting
from governance_cadence import GovernanceCadence
from jira_connector import JiraConnector

print("✅ All modules imported successfully!")

## 2. Generate Templates (First Time Setup)

In [ ]:
from create_templates import main as create_templates_main

create_templates_main()
print("\n✅ Templates created in templates/ folder")

## 3. Option A: Connect to Jira and Pull Data (Optional)

In [ ]:
# Initialize Jira Connector
jira = JiraConnector()

# Try to connect (will skip if credentials not configured)
if jira.connect():
    print("✅ Connected to Jira successfully!")
    
    # Pull data from Jira
    jira.export_to_master_register()
    jira.export_actions()
    
    print("✅ Jira data exported to data/input/")
else:
    print("⚠️ Jira not configured. Using manual data instead.")
    print("To use Jira: Edit .env file with your credentials")

## 4. Generate Master Register

In [ ]:
# Initialize Master Register
register = MasterRegister()

# Load data from all sources
register.load_register()

# Display summary
stats = register.get_summary_stats()
print(f"\n📊 Master Register Summary:")
print(f"   Total Items: {stats['total_items']}")

if stats['by_tpr_area']:
    print(f"\n   By TPR Area:")
    for area, count in stats['by_tpr_area'].items():
        print(f"     {area}: {count}")

# Export to Excel
output_file = register.export_to_excel()
print(f"\n✅ Master Register exported: {output_file}")

# Display the data
if not register.register_data.empty:
    display(register.register_data.head())
else:
    print("\n⚠️ No data in Master Register. Add data to data/input/ folder.")

## 5. Generate Action Tracker

In [ ]:
# Initialize Action Tracker
tracker = ActionTracker()

# Load actions from all sources
tracker.load_tracker()

# Display summary
stats = tracker.get_summary_stats()
print(f"\n📊 Action Tracker Summary:")
print(f"   Total Actions: {stats['total_actions']}")
print(f"   Overdue: {stats['overdue_count']}")
print(f"   Due This Week: {stats['due_this_week']}")
print(f"   Critical Priority: {stats['critical_count']}")

# Export to Excel
output_file = tracker.export_to_excel()
print(f"\n✅ Action Tracker exported: {output_file}")

# Display overdue actions
overdue = tracker.get_overdue_actions()
if not overdue.empty:
    print(f"\n⚠️ Overdue Actions ({len(overdue)}):")
    display(overdue[['Action ID', 'Description', 'Owner', 'Due Date', 'Priority']])
else:
    print("\n✅ No overdue actions!")

# Display all actions
if not tracker.actions_data.empty:
    print("\n📋 All Actions:")
    display(tracker.actions_data.head())

## 6. Generate TPR Report

In [ ]:
# Initialize TPR Reporting
tpr = TPRReporting()

# Load data
tpr.load_data()

# Analyze all TPR areas
print("\n📊 Analyzing TPR Areas...\n")
for tpr_area in tpr.tpr_areas:
    area_name = tpr_area['name']
    analysis = tpr.analyze_tpr_area(area_name)
    tpr.report_data[area_name] = analysis
    
    # Display summary
    status_emoji = "🟢" if analysis['status'] == 'Green' else ("🟡" if analysis['status'] == 'Amber' else "🔴")
    print(f"{status_emoji} {area_name}:")
    print(f"   Status: {analysis['status']}")
    print(f"   Initiatives: {analysis['total_initiatives']} (Active: {analysis['active_initiatives']})")
    print(f"   Actions: {analysis['total_actions']} (Open: {analysis['open_actions']})")
    print()

# Generate executive summary
exec_summary = tpr.generate_executive_summary()
print("\n📊 Executive Summary:")
display(exec_summary)

# Export to Excel
output_file = tpr.export_report()
print(f"\n✅ TPR Report exported: {output_file}")

## 7. Generate Governance Cadence

In [ ]:
# Initialize Governance Cadence
cadence = GovernanceCadence()

# Generate 6-month schedule
schedule = cadence.generate_meeting_schedule(months_ahead=6)

# Display upcoming meetings
upcoming = cadence.get_upcoming_meetings(14)
print("\n📅 Upcoming Meetings (Next 2 Weeks):")
if not upcoming.empty:
    display(upcoming[['Date', 'Meeting Name', 'Time', 'Attendees']])
else:
    print("   No meetings in next 2 weeks")

# Display this month's meetings
this_month = cadence.get_this_month_meetings()
print(f"\n📅 This Month's Meetings ({len(this_month)}):")
if not this_month.empty:
    display(this_month[['Date', 'Meeting Name', 'Time']])

# Export to Excel
output_file = cadence.export_to_excel()
print(f"\n✅ Governance Cadence exported: {output_file}")

## 8. View All Generated Reports

In [ ]:
import os

output_path = Path('data/output')

print("\n📁 Generated Reports:")
print("=" * 60)

if output_path.exists():
    files = list(output_path.glob('*.xlsx'))
    if files:
        for file in sorted(files):
            size = file.stat().st_size / 1024  # KB
            print(f"   ✅ {file.name} ({size:.1f} KB)")
    else:
        print("   ⚠️ No reports generated yet")
else:
    print("   ⚠️ Output directory not found")

print("=" * 60)
print("\nReports are available in: data/output/")
print("Open them in Excel or Google Sheets to view!")

## 9. Quick Analysis: Visualize TPR Status

In [ ]:
import matplotlib.pyplot as plt

# Get TPR status summary
if tpr.report_data:
    status_counts = {'Green': 0, 'Amber': 0, 'Red': 0}
    
    for area_name, analysis in tpr.report_data.items():
        status = analysis.get('status', 'Green')
        status_counts[status] += 1
    
    # Create pie chart
    colors = {'Green': '#90EE90', 'Amber': '#FFD700', 'Red': '#FF6B6B'}
    plt.figure(figsize=(8, 6))
    plt.pie(
        status_counts.values(),
        labels=status_counts.keys(),
        colors=[colors[k] for k in status_counts.keys()],
        autopct='%1.0f%%',
        startangle=90
    )
    plt.title('TPR Areas by Status (RAG)', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.show()
    
    print(f"\n📊 Status Summary:")
    print(f"   🟢 Green: {status_counts['Green']} areas")
    print(f"   🟡 Amber: {status_counts['Amber']} areas")
    print(f"   🔴 Red: {status_counts['Red']} areas")
else:
    print("⚠️ No TPR data available for visualization")

## 10. Add Manual Data Example

In [ ]:
# Example: Add a new item to Master Register
register_example = MasterRegister()
register_example.load_register()

new_item = {
    'ID': 'TECH-999',
    'Name': 'Example Initiative from Notebook',
    'Type': 'Initiative',
    'TPR Area': 'Architecture',
    'Owner': 'Your Name',
    'Status': 'In Progress',
    'Priority': 'High',
    'Start Date': '2026-06-01',
    'End Date': '2026-12-31',
    'Budget': '100000',
    'Risk Level': 'Medium',
    'Description': 'Example initiative added from Jupyter notebook',
    'Source': 'Manual Entry'
}

register_example.add_item(new_item)
print("✅ Added example item to Master Register")
print("\nTo save it, run:")
print("register_example.export_to_excel('data/output/example_register.xlsx')")

## 🎯 Summary

You can now:
- Run each module independently
- View results inline
- Add data programmatically
- Visualize data
- Export reports

## Next Steps:
1. Customize the configuration files
2. Add your real data
3. Run the cells to generate reports
4. Share reports with your team

## Resources:
- Full documentation in `docs/` folder
- Quick reference: `QUICK_REFERENCE.md`
- Setup guide: `docs/SETUP_GUIDE.md`